# Khmer TTS & Voice Cloning — Universal Runner

End-to-end pipeline (data prep → Khmer base model → voice clone → eval → serve).
Runs unmodified on **Kaggle** (free T4 GPU), **Google Colab**, or a **local machine**
with a GPU — the environment is auto-detected in Section 1, no manual edits needed
beyond the CONFIG cell.

## Before you run
1. **Enable a GPU**: Kaggle → Settings ▸ Accelerator ▸ GPU T4 x2; Colab → Runtime ▸
   Change runtime type ▸ GPU; local → any CUDA-capable GPU (CPU also works for the
   smoke-test-sized defaults, just slower).
2. Make sure you have internet access (needed to pip-install and download models).
3. Make your repo available to the notebook — pick ONE (see CONFIG cell):
   - **Local**: leave `WORKDIR`/`GITHUB_URL`/`DATASET_PATH` empty and run the notebook
     from inside the repo — it reuses the current directory.
   - **Colab**: set `GITHUB_URL` to your repo's git URL (auto-clones into `/content`).
   - **Kaggle**: push to GitHub and set `GITHUB_URL`, or **Add data ▸ Upload** the repo
     as a Kaggle Dataset and set `DATASET_PATH`.
4. (Only if the DDD dataset is gated) add an `HF_TOKEN` secret — Kaggle: Add-ons ▸
   Secrets; Colab: the 🔑 Secrets panel; local: `export HF_TOKEN=...` or
   `huggingface-cli login`. A free Hugging Face account is enough.

> **Kaggle session limits:** 12h max per run, ~30h GPU/week, 20GB in `/kaggle/working`
> (saved when you *Save Version*). **Colab** free tier: ~12h max, session can be
> reclaimed if idle. The training defaults below are deliberately small so a run
> finishes in one session; scale up + resume across sessions once it works.

## 0 · Configuration — edit this cell, then Run All

In [ ]:
# ============================= CONFIG =============================
# --- Where your repo code comes from ---
# Leave all three blank to run in-place (recommended for a local machine already
# inside the repo). On Kaggle/Colab, set ONE of GITHUB_URL / DATASET_PATH.
GITHUB_URL   = ""   # e.g. "https://github.com/you/khmer-voice-clone.git"
DATASET_PATH = ""   # e.g. "/kaggle/input/khmer-voice-clone" (Kaggle Dataset upload)

# --- Working dir ---
# Leave "" to auto-pick based on detected environment:
#   Kaggle -> /kaggle/working/khmer-voice-clone
#   Colab  -> /content/khmer-voice-clone
#   Local  -> current directory (this notebook is assumed to sit inside the repo)
WORKDIR = ""

# --- Data / model sources (all free downloads) ---
DDD_DATASET    = "DDD-Cambodia/khmer-speech-dataset"  # ~70h Khmer, multi-speaker
DDD_SPLIT      = "train"
BASE_CKPT_REPO = "fishaudio/openaudio-s1-mini"    # free pretrained Fish Speech checkpoint

# --- Run size knobs ---
# Start SMALL to prove the whole pipeline runs, then raise these.
MAX_SAMPLES   = 200     # cap DDD clips for a first smoke run; set None for the full dataset
STAGE1_STEPS  = 2000    # Khmer base steps  (target ~20000 across multiple sessions)
STAGE2_STEPS  = 1000    # voice-clone steps (target ~3000)

# --- Your own voice (Stage 2). Provide a JSONL of {audio_path, text} lines. ---
# Upload your recordings (Kaggle Dataset / Colab Drive / local path) and point here;
# leave as-is to skip Stage 2.
MY_VOICE_MANIFEST = ""  # e.g. "/kaggle/input/my-voice/my_voice_raw.jsonl"
MY_VOICE_SPEAKER  = "my_voice"

# --- Collaborative relay (you + friends pool GPUs on one shared model) ---
ENABLE_RELAY = False   # set True to stream a shard + sync checkpoints via HF
HF_CKPT_REPO = "your-username/khmer-tts-relay"  # shared HF model repo (created if missing)
TRAINER_ID   = "friendA"   # UNIQUE per person (friendA / friendB / ...)
NUM_SHARDS   = 2       # how many people are collaborating
SHARD_INDEX  = 0       # YOUR shard: 0-based, must be unique per person
TAKE_PER_SESSION = 4000  # clips to stream from your shard this session
# =================================================================
print('Config loaded.')

## 1 · Environment & GPU check

In [ ]:
import subprocess, sys, os

# Detect which platform we're running on. Check Colab first: `import google.colab`
# succeeding is authoritative (that package only exists in Colab's environment),
# whereas checking for a /kaggle path is a weaker heuristic that can false-positive.
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
IN_KAGGLE = (not IN_COLAB) and os.path.exists('/kaggle')
IN_LOCAL = not IN_KAGGLE and not IN_COLAB
ENV_NAME = 'Colab' if IN_COLAB else ('Kaggle' if IN_KAGGLE else 'Local')
print('Environment:', ENV_NAME)

import torch
print('Python :', sys.version.split()[0])
print('Torch  :', torch.__version__)
print('CUDA   :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU    :', torch.cuda.get_device_name(0), f'({torch.cuda.device_count()} device(s))')
    print(subprocess.run(['nvidia-smi','--query-gpu=memory.total,memory.free','--format=csv'],
                         capture_output=True, text=True).stdout)
else:
    print('!! No GPU detected. Training will be very slow on CPU.',
          '' if IN_LOCAL else 'Enable the GPU accelerator in this platform\'s settings.')

## 2 · Get the code (your repo + Fish Speech engine)

In [ ]:
import os, shutil, subprocess

# 2a. Resolve WORKDIR for the detected environment if left blank.
if not WORKDIR:
    if IN_KAGGLE:
        WORKDIR = '/kaggle/working/khmer-voice-clone'
    elif IN_COLAB:
        # Mount Drive so the repo (code + downloaded data + checkpoints) survives a
        # disconnect/reclaim instead of vanishing with ephemeral /content storage.
        from google.colab import drive
        drive.mount('/content/drive')
        WORKDIR = '/content/drive/MyDrive/khmer-voice-clone'
    else:
        WORKDIR = os.getcwd()  # assume local run happens from inside the repo

# 2b. Bring in your repo -> WORKDIR
REPO_MARKER = os.path.join(WORKDIR, 'khmer_tts')
if os.path.isdir(REPO_MARKER):
    print('WORKDIR already has the repo, reusing:', WORKDIR)
elif GITHUB_URL:
    subprocess.run(['git', 'clone', '--depth', '1', GITHUB_URL, WORKDIR], check=True)
elif DATASET_PATH and os.path.isdir(DATASET_PATH):
    shutil.copytree(DATASET_PATH, WORKDIR, dirs_exist_ok=True)
    print('Copied repo from dataset ->', WORKDIR)
elif IN_LOCAL:
    raise SystemExit(
        f'WORKDIR ({WORKDIR}) does not look like the repo root (no khmer_tts/ found). '
        'Run this notebook from inside the repo, or set GITHUB_URL/DATASET_PATH.'
    )
else:
    raise SystemExit('Set GITHUB_URL or add the repo as a Dataset and set DATASET_PATH.')

os.chdir(WORKDIR)          # all scripts assume repo root is the cwd
print('cwd =', os.getcwd())

# 2c. Clone the Fish Speech engine into vendor/ (the actual TTS model code)
FISH_DIR = os.path.join(WORKDIR, 'vendor', 'fish-speech')
if not os.path.isdir(os.path.join(FISH_DIR, 'fish_speech')):
    shutil.rmtree(FISH_DIR, ignore_errors=True)
    subprocess.run(['git', 'clone', '--depth', '1',
                    'https://github.com/fishaudio/fish-speech', FISH_DIR], check=True)
print('Fish Speech at:', FISH_DIR)

## 3 · Install dependencies

Kaggle already ships CUDA-enabled PyTorch, so we install the lighter deps plus the
Fish Speech package itself. The optional VAD/denoise deps make the audio cleanup steps
real (they no-op/copy-through if missing).

In [ ]:
import importlib.util
_torch_ok = False
if importlib.util.find_spec('torch') is not None:
    import torch as _torch_probe
    _torch_ok = _torch_probe.cuda.is_available()
    del _torch_probe

if _torch_ok:
    # A working CUDA torch is already installed (typical on a local machine) --
    # torchaudio's own dependency resolution can otherwise drag in a fresh torch wheel
    # and silently clobber it with a CPU-only/mismatched-CUDA build.
    print('Working CUDA torch already present -- installing torchaudio --no-deps to avoid clobbering it.')
    reqs = [l for l in open('requirements.txt') if not l.strip().lower().startswith('torchaudio')]
    open('/tmp/requirements_no_torchaudio.txt', 'w').writelines(reqs)
    get_ipython().system('pip install -q -r /tmp/requirements_no_torchaudio.txt')
    get_ipython().system('pip install -q --no-deps "torchaudio>=2.2"')
else:
    get_ipython().system('pip install -q -r requirements.txt')

# Optional but recommended for real audio cleanup
get_ipython().system('pip install -q deepfilternet silero-vad huggingface_hub')
# Install the Fish Speech engine in editable mode
get_ipython().system('pip install -q -e vendor/fish-speech')
print('Dependencies installed.')

## 4 · Secrets & base checkpoint download (free)

In [ ]:
import os

# HF token, tried in order: Kaggle Secrets -> Colab Secrets -> local env var ->
# cached `huggingface-cli login` credential. Only needed for gated datasets/models --
# safe to skip if yours are public.
HF_TOKEN = ''
if IN_KAGGLE:
    try:
        from kaggle_secrets import UserSecretsClient
        HF_TOKEN = UserSecretsClient().get_secret('HF_TOKEN')
        print('HF_TOKEN loaded from Kaggle Secrets.')
    except Exception as e:
        print('No HF_TOKEN in Kaggle Secrets (fine if your dataset/model is public):', e)
elif IN_COLAB:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN') or ''
        if HF_TOKEN:
            print('HF_TOKEN loaded from Colab Secrets.')
    except Exception as e:
        print('No HF_TOKEN in Colab Secrets (fine if your dataset/model is public):', e)

if not HF_TOKEN:
    HF_TOKEN = os.environ.get('HF_TOKEN', '')
    if HF_TOKEN:
        print('HF_TOKEN loaded from local environment.')

if not HF_TOKEN:
    try:
        from huggingface_hub import get_token
        HF_TOKEN = get_token() or ''
        if HF_TOKEN:
            print('HF_TOKEN loaded from cached `huggingface-cli login`.')
    except Exception:
        pass

if HF_TOKEN:
    os.environ['HF_TOKEN'] = HF_TOKEN
    os.environ['HUGGING_FACE_HUB_TOKEN'] = HF_TOKEN
    from huggingface_hub import login
    login(token=HF_TOKEN)
else:
    print('No HF_TOKEN found anywhere -- fine if your dataset/model is public.')

In [ ]:
# Download the free pretrained Fish Speech checkpoint into checkpoints/
from huggingface_hub import snapshot_download
import os
CKPT_DIR = os.path.join(WORKDIR, 'checkpoints', 'openaudio-s1-mini')
os.makedirs(CKPT_DIR, exist_ok=True)
snapshot_download(repo_id=BASE_CKPT_REPO, local_dir=CKPT_DIR,
                  token=os.environ.get('HF_TOKEN'))
print('Base checkpoint at:', CKPT_DIR)
print(os.listdir(CKPT_DIR))

## 4.5 · Collaborative relay — pull best checkpoint + stream your shard

**Only runs when `ENABLE_RELAY = True`.** Each person sets a unique `TRAINER_ID`
and `SHARD_INDEX`. This cell: creates/opens the shared HF repo, grabs an advisory
lock, pulls the **best** checkpoint anyone has pushed, and streams a small chunk of
**your** shard (whole speakers are assigned to shards, so no overlap). Then the normal
data cells below process just that chunk, training resumes from the pulled checkpoint,
and the publish cell pushes the result back for your friend to continue.

> Take **turns** — the lock guards against accidental simultaneous training, but it is
> advisory (HF has no atomic locking). One person trains, pushes, then the next pulls.

In [ ]:
RESUME_CKPT = None
if ENABLE_RELAY:
    import os
    from khmer_tts.collab import HFCheckpointRelay, stream_shard_to_disk
    relay = HFCheckpointRelay(HF_CKPT_REPO, token=os.environ.get('HF_TOKEN'))
    relay.ensure_repo()
    ok, lk = relay.acquire_lock(TRAINER_ID, ttl_sec=6*3600)
    if not ok:
        raise SystemExit(f"Repo is locked by {lk.get('owner')} until it expires — coordinate turns.")
    best = relay.best_checkpoint()
    print('best checkpoint so far:', best)
    RESUME_CKPT = relay.pull_best('checkpoints/_resume') if best else None
    print('resuming from:', RESUME_CKPT or '(none yet — will start from the base checkpoint)')
    # Stream a small chunk of THIS person's shard into the raw manifest.
    open('data/manifests/ddd_raw.jsonl', 'w').close()
    n = stream_shard_to_disk(DDD_DATASET, DDD_SPLIT, NUM_SHARDS, SHARD_INDEX,
                             TAKE_PER_SESSION, 'data/raw/ddd/audio',
                             'data/manifests/ddd_raw.jsonl', token=os.environ.get('HF_TOKEN'))
    print(f'streamed {n} clips for shard {SHARD_INDEX} (trainer {TRAINER_ID})')
else:
    print('Relay disabled — single-person mode. Section 5 will download normally.')

# ===== STAGE 1: BUILD THE KHMER BASE MODEL =====

## 5 · Data pipeline (download → QC → clean → normalize → split)

Each step writes a manifest under `data/manifests/`. Runs your existing numbered
scripts unchanged. First pass uses `MAX_SAMPLES` for a quick smoke test.

In [ ]:
import os
# 01 download DDD (skipped in relay mode — section 4.5 already streamed your shard)
if ENABLE_RELAY:
    print('Relay mode: using the streamed shard manifest, skipping full download.')
else:
    open('data/manifests/ddd_raw.jsonl','w').close() if os.path.exists('data/manifests/ddd_raw.jsonl') else None
    cap = f'--max_samples {MAX_SAMPLES}' if MAX_SAMPLES else ''
    get_ipython().system(f'python scripts/01_download_ddd.py --dataset {DDD_DATASET} --split {DDD_SPLIT} --out_dir data/raw/ddd {cap}')

In [ ]:
# 02 consolidate + verify
!python scripts/02_export_audio.py --inputs data/manifests/ddd_raw.jsonl --output data/manifests/ddd_raw_merged.jsonl
# 03 quality-control grading (A/B/C/D)
!python scripts/03_audio_qc.py --manifest data/manifests/ddd_raw_merged.jsonl --output data/manifests/ddd_qc.jsonl

In [ ]:
# 04 selective denoise (B-grade only)
!python scripts/04_denoise.py --manifest data/manifests/ddd_qc.jsonl --output_dir data/processed/ddd_denoised --output_manifest data/manifests/ddd_denoised.jsonl
# 05 VAD trim silence
!python scripts/05_vad_trim.py --manifest data/manifests/ddd_denoised.jsonl --output_dir data/processed/ddd_vad --output_manifest data/manifests/ddd_vad.jsonl
# 06 loudness normalize + 24kHz resample
!python scripts/06_loudness_normalize.py --manifest data/manifests/ddd_vad.jsonl --output_dir data/processed/ddd_24k --output_manifest data/manifests/ddd_clean.jsonl

In [ ]:
# 07 Khmer text normalization
!python scripts/07_normalize_khmer_text.py --manifest data/manifests/ddd_clean.jsonl --output data/manifests/ddd_normalized.jsonl
# 08 train/valid/test split
!python scripts/08_make_splits.py --manifest data/manifests/ddd_normalized.jsonl --out_prefix data/manifests/ddd
# validate before spending GPU time
!python scripts/validate_manifest.py data/manifests/ddd_train.jsonl

In [ ]:
# 09 convert to Fish Speech speaker-folder format (--copy: Kaggle-safe, no symlinks)
!python scripts/09_convert_to_fish_format.py --manifest data/manifests/ddd_train.jsonl --out_dir data/fish/khmer_base --copy

## 6 · Train the Khmer base model (Stage 1)

> **Note on capacity:** `scripts/10` fine-tunes with a small LoRA (rank 8). Teaching a
> whole new *language* often wants more capacity than that. For a real base model,
> edit `scripts/10_train_fish_khmer_base.sh` to use `r_32_alpha_64` (or a full
> fine-tune) and raise `STAGE1_STEPS` toward 20000 across sessions. The Fish CLI flags
> can drift between versions — if a step errors, check `vendor/fish-speech/tools/`.

We override the step count via an env var the cell passes through.

In [ ]:
import os, subprocess
os.environ['STAGE1_MAX_STEPS'] = str(STAGE1_STEPS)
# The script hardcodes trainer.max_steps; sed it to our knob.
subprocess.run(['sed','-i',
    f"s/trainer.max_steps=[0-9]*/trainer.max_steps={STAGE1_STEPS}/",
    'scripts/10_train_fish_khmer_base.sh'], check=True)
# Relay mode: resume from the pulled checkpoint instead of the pretrained base.
if ENABLE_RELAY and RESUME_CKPT:
    subprocess.run(['sed','-i',
        f's#^CHECKPOINT_DIR=.*#CHECKPOINT_DIR="{RESUME_CKPT}"#',
        'scripts/10_train_fish_khmer_base.sh'], check=True)
    print('resuming training from', RESUME_CKPT)
!bash scripts/10_train_fish_khmer_base.sh

## 6.5 · Publish your checkpoint to the shared HF repo (relay mode)

Uploads the checkpoint you just trained and records it in the registry with its
validation loss, so `best_checkpoint()` stays correct. Then releases the lock — your
friend pulls it next and continues from here.

In [ ]:
if ENABLE_RELAY:
    from khmer_tts.collab import read_val_loss
    prev = relay.best_checkpoint()
    prev_step = prev['step'] if prev else 0
    new_step = prev_step + STAGE1_STEPS          # cumulative steps across everyone
    vloss = read_val_loss('models/khmer_base')   # None if logs weren't parseable
    print('publishing: step', new_step, 'val_loss', vloss)
    entry = relay.publish('models/khmer_base', step=new_step, val_loss=vloss,
                          trainer_id=TRAINER_ID, shard_index=SHARD_INDEX)
    relay.release_lock(TRAINER_ID)
    print('pushed', entry['path'], 'to', HF_CKPT_REPO)
    print('Your friend can now Run All (with their own SHARD_INDEX) to continue.')
else:
    print('Relay disabled — nothing to publish.')

## 7 · Sanity-check the base model (pronunciation)

In [ ]:
!python scripts/13_generate_eval_samples.py --model_dir models/khmer_base --speaker default --sentences eval/test_sentences_km.txt --out_dir outputs/eval/khmer_base_v1
import IPython.display as ipd, glob
for w in sorted(glob.glob('outputs/eval/khmer_base_v1/*.wav'))[:3]:
    print(w); ipd.display(ipd.Audio(w))

# ===== STAGE 2: CLONE YOUR OWN VOICE =====

## 8 · Convert & train on your voice

Set `MY_VOICE_MANIFEST` in the config cell to a JSONL of `{"audio_path": ..., "text": ...}`.
Aim for 30 min–3 hr, same room/mic. Stage 2 uses a **low-rank LoRA on a frozen base** so
it learns your voice without forgetting Khmer.

In [ ]:
import os
if not MY_VOICE_MANIFEST or not os.path.exists(MY_VOICE_MANIFEST):
    print('Skipping Stage 2 — set MY_VOICE_MANIFEST to your recordings JSONL to enable.')
else:
    os.environ['STAGE2_MAX_STEPS'] = str(STAGE2_STEPS)
    !python scripts/11_convert_my_voice_to_fish.py --manifest {MY_VOICE_MANIFEST} --speaker_id {MY_VOICE_SPEAKER} --out_dir data/fish/my_voice
    !sed -i 's/trainer.max_steps=[0-9]*/trainer.max_steps='$STAGE2_MAX_STEPS'/' scripts/12_train_fish_my_voice.sh
    !bash scripts/12_train_fish_my_voice.sh

## 9 · Evaluate the voice clone

In [ ]:
import os, glob, IPython.display as ipd
if os.path.isdir('models/my_voice') and os.listdir('models/my_voice'):
    !python scripts/13_generate_eval_samples.py --model_dir models/my_voice --speaker {MY_VOICE_SPEAKER} --sentences eval/test_sentences_km.txt --out_dir outputs/eval/my_voice_v1
    for w in sorted(glob.glob('outputs/eval/my_voice_v1/*.wav'))[:3]:
        print(w); ipd.display(ipd.Audio(w))
else:
    print('No voice-clone model yet (Stage 2 skipped).')

## 10 · Quick synthesis test (inference)

In [ ]:
import os, IPython.display as ipd
from khmer_tts.inference.fish_backend import FishSpeechBackend
from khmer_tts.text.normalize import normalize_khmer_text

model_dir = 'models/my_voice' if os.path.isdir('models/my_voice') and os.listdir('models/my_voice') else 'models/khmer_base'
backend = FishSpeechBackend(model_dir=model_dir, device='cuda')
text = normalize_khmer_text('សួស្តី ខ្ញុំអាចជួយអ្នកបាន។ តម្លៃគឺ $10 នៅឆ្នាំ 2026។')
out = 'outputs/api/demo.wav'
backend.synthesize_long_text(text=text, output_path=out, speaker='default')
print('normalized:', text)
ipd.display(ipd.Audio(out))

## 11 · Save your trained models

Trained models are zipped into `WORKDIR/artifacts/`. What to do next depends on
where you ran this:
- **Kaggle**: click **Save Version** to persist `/kaggle/working` (includes the zips).
- **Colab**: copy the zips to Google Drive (`from google.colab import drive;
  drive.mount('/content/drive')`) before the session ends, or download them directly.
- **Local**: the zips (and the unzipped `models/`) are already on disk under
  `WORKDIR` — nothing further needed.

In [ ]:
import shutil, os

artifacts_dir = os.path.join(WORKDIR, 'artifacts')
os.makedirs(artifacts_dir, exist_ok=True)
for m in ['models/khmer_base', 'models/my_voice']:
    if os.path.isdir(m) and os.listdir(m):
        name = os.path.basename(m)
        shutil.make_archive(os.path.join(artifacts_dir, name), 'zip', m)
        print('zipped', m, '->', os.path.join(artifacts_dir, name + '.zip'))

if IN_KAGGLE:
    print('Now click *Save Version* to persist /kaggle/working.')
elif IN_COLAB:
    print('Download the zips from', artifacts_dir, 'or copy them to Drive before the session ends.')
else:
    print('Artifacts saved under', artifacts_dir)